In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.tsa.seasonal import STL

from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().resolve().parent))
print(sys.executable)

from src.metrics import *
print("ok")

/Users/yuanxinyi/Desktop/panel-forcast-kit/.venv/bin/python
ok


In [11]:
project_root = Path.cwd().resolve().parent if "panel-forcast-kit" in str(Path.cwd()) else Path.cwd()

processed_dir = project_root / "data" / "processed"

X_train = pd.read_csv(processed_dir / "X_train.csv")
y_train = pd.read_csv(processed_dir / "y_train.csv")
X_test = pd.read_csv(processed_dir / "X_test.csv")
y_test = pd.read_csv(processed_dir / "y_test.csv")

# 统一为 Series / DataFrame 的形式
X_train = X_train.copy()
y_train = y_train.iloc[:, 0].copy()
X_test = X_test.copy()
y_test = y_test.iloc[:, 0].copy()

print("Loaded:")
print(f"X_train shape: {X_train.shape}, \n{X_train.head(1)}")
print(f"\ny_train shape: {y_train.shape}, \n{y_train.head(1)}")
print(f"\nX_test shape: {X_test.shape}, \n{X_test.head(1)}")
print(f"\ny_test shape: {y_test.shape}, \n{y_test.head(1)}")

Loaded:
X_train shape: (853000, 11), 
   store  item  day  month  is_weekend  is_holiday  lag_1  lag_7  lag_14  \
0      1     1   29      1           0           0   11.0    7.0     5.0   

   lag_28  rolling_mean_7  
0    13.0       10.428571  

y_train shape: (853000,), 
0    6
Name: sales, dtype: int64

X_test shape: (46000, 11), 
   store  item  day  month  is_weekend  is_holiday  lag_1  lag_7  lag_14  \
0      1     1    1     10           1           0   30.0   24.0    30.0   

   lag_28  rolling_mean_7  
0    23.0       23.428571  

y_test shape: (46000,), 
0    21
Name: sales, dtype: int64


# Naive Baseline

In [3]:
# Baseline: direct prediction and evaluation on the already split test set
baseline_cols = {
    "Same period last week (lag7)": "lag_7",
    "7-day moving average (MA7)": "rolling_mean_7"
}

print(f"\n=== Baseline performance on the test set (sample count: {len(X_test)}) ===")
baseline_results = {}
for name, col in baseline_cols.items():
    baseline_results[name] = evaluate(y_test, X_test[col])
    mae, rmse, wape = baseline_results[name]
    print(f"{name:20s}  MAE: {mae:6.3f}   RMSE: {rmse:6.3f}   WAPE: {wape:6.2%}")


=== Baseline performance on the test set (sample count: 46000) ===
Same period last week (lag7)  MAE:  9.076   RMSE: 12.051   WAPE: 16.60%
7-day moving average (MA7)  MAE:  8.712   RMSE: 11.463   WAPE: 15.93%


# Tree Ensemble Baseline

## Poisson GBDT

非负销量场景常见强基线

In [27]:
from sklearn.ensemble import HistGradientBoostingRegressor

model1 = HistGradientBoostingRegressor(
    loss="poisson",
    learning_rate=0.05,
    max_iter=600,
    max_depth=8,
    min_samples_leaf=40,
    l2_regularization=1.0,
    random_state=42
)

model1.fit(X_train, y_train)
pred1 = model1.predict(X_test)

print("=== Model 1 Performance: Poisson GBDT ===")
mae, rmse, wmape = evaluate(y_test, pred1)
print(f"MAE: {mae:6.3f}   RMSE: {rmse:6.3f}   WAPE: {wmape:6.2%}")

=== Model 1 Performance: Poisson GBDT ===
MAE:  6.296   RMSE:  8.153   WAPE: 11.51%


## XGboost

In [20]:
from xgboost import XGBRegressor
import numpy as np

model_xgb_default = XGBRegressor()
model_xgb_default.fit(X_train, y_train)

pred = model_xgb_default.predict(X_test)
mae, rmse, wmape = evaluate(y_test, pred)
print(f"默认参数 XGBoost  MAE: {mae:.3f}  RMSE: {rmse:.3f}  WAPE: {wmape:.2%}")

默认参数 XGBoost  MAE: 6.356  RMSE: 8.239  WAPE: 11.63%


In [25]:
import pandas as pd
feature_cols = X_train.columns
imp = pd.Series(model_xgb_default.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(imp)

lag_7             0.496245
rolling_mean_7    0.318821
lag_14            0.064242
is_weekend        0.045998
lag_28            0.040467
lag_1             0.014589
month             0.013679
day               0.004662
is_holiday        0.000501
item              0.000428
store             0.000369
dtype: float32


In [ ]:
# xgboost random search
from xgboost import XGBRegressor

model_xgb = XGBRegressor(
    objective="count:poisson",   # 非负销量场景更匹配
    eval_metric="poisson-nloglik",
    learning_rate=0.05,
    n_estimators=600,
    max_depth=8,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

model_xgb.fit(X_train, y_train)
pred_xgb = model_xgb.predict(X_test)

print("=== Model X Performance: XGBoost ===")
mae, rmse, wmape = evaluate(y_test, pred_xgb)
print(f"MAE: {mae:6.3f}   RMSE: {rmse:6.3f}   WAPE: {wmape:6.2%}")

=== Model X Performance: XGBoost ===
MAE:  6.289   RMSE:  8.145   WAPE: 11.50%


## Light GBM

In [15]:
import lightgbm as lgb

model2 = lgb.LGBMRegressor(
    objective="regression",
    learning_rate=0.05,
    n_estimators=600,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=20,
    random_state=42,
    n_jobs=-1
)

model2.fit(X_train, y_train)
pred2 = model2.predict(X_test)

print("=== Model 2 Performance: LightGBM ===")
mae, rmse, wmape = evaluate(y_test, pred2)
print(f"MAE: {mae:6.3f}   RMSE: {rmse:6.3f}   WAPE: {wmape:6.2%}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.018705 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1123
[LightGBM] [Info] Number of data points in the train set: 853000, number of used features: 11
[LightGBM] [Info] Start training from score 52.492708
=== Model 2 Performance: LightGBM ===
MAE:  6.298   RMSE:  8.153   WAPE: 11.52%


## CatBoost

In [13]:
from catboost import CatBoostRegressor

model3 = CatBoostRegressor(
    loss_function="RMSE",
    iterations=600,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3.0,
    random_seed=42,
    verbose=0,
    od_type="Iter",
    od_wait=50
)

model3.fit(X_train, y_train)
pred3 = model3.predict(X_test)

print("=== Model 3 Performance: CatBoost ===")
mae, rmse, wmape = evaluate(y_test, pred3)
print(f"MAE: {mae:6.3f}   RMSE: {rmse:6.3f}   WAPE: {wmape:6.2%}")

=== Model 3 Performance: CatBoost ===
MAE:  6.279   RMSE:  8.127   WAPE: 11.48%


# Linear Regularized Baseline

## ElasticNetCV

In [12]:
print("feature col:", X_train.columns.tolist())
print(X_train.head(5))

feature col: ['store', 'item', 'day', 'month', 'is_weekend', 'is_holiday', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7']
   store  item  day  month  is_weekend  is_holiday  lag_1  lag_7  lag_14  \
0      1     1   29      1           0           0   11.0    7.0     5.0   
1      1     1   30      1           0           0    6.0    9.0     7.0   
2      1     1   31      1           0           0    9.0    8.0    16.0   
3      1     1    1      2           0           0   13.0   14.0     7.0   
4      1     1    2      2           1           0   11.0   12.0    18.0   

   lag_28  rolling_mean_7  
0    13.0       10.428571  
1    11.0       10.285714  
2    14.0       10.285714  
3    13.0       11.000000  
4    10.0       10.571429  


## Feature Scaling

In [8]:
from sklearn.preprocessing import StandardScaler

# 你当前数据中的类别特征
cat_cols = ["store", "item", "day", "month", "is_weekend", "is_holiday"]

# 你当前数据中的连续特征
num_cols = ["lag_1", "lag_7", "lag_14", "lag_28", "rolling_mean_7"]

# 复制原始数据，避免原表被改坏
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# 对连续特征做标准化
scaler = StandardScaler()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

# 这里不动类别特征
# 也就是 store/item/day/month/is_weekend/is_holiday 保持原值
print(X_train_scaled.head())
print(X_train_scaled.shape)

   store  item  day  month  is_weekend  is_holiday     lag_1     lag_7  \
0      1     1   29      1           0           0 -1.438261 -1.574183   
1      1     1   30      1           0           0 -1.611687 -1.504774   
2      1     1   31      1           0           0 -1.507631 -1.539478   
3      1     1    1      2           0           0 -1.368890 -1.331252   
4      1     1    2      2           1           0 -1.438261 -1.400661   

     lag_14    lag_28  rolling_mean_7  
0 -1.640110 -1.355135       -1.554257  
1 -1.570652 -1.424701       -1.559546  
2 -1.258091 -1.320352       -1.559546  
3 -1.570652 -1.355135       -1.533105  
4 -1.188633 -1.459485       -1.548969  
(853000, 11)


## ElasticCV

In [9]:
from sklearn.linear_model import ElasticNetCV

model4_enet = ElasticNetCV(
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
    cv=5,
    max_iter=20000,
    random_state=0
)
model4_enet.fit(X_train_scaled, y_train)

print("best alpha:   ", model4_enet.alpha_)
print("best l1_ratio:", model4_enet.l1_ratio_)

pred4_enet = model4_enet.predict(X_test_scaled)
mae, rmse, wmape = evaluate(y_test, pred4_enet)

best alpha:    0.026802434146355872
best l1_ratio: 1.0


In [10]:
coef = pd.Series(model4_enet.coef_, index=X_train.columns)
print("非零:", (coef != 0).sum(), "/", len(coef))
print(coef[coef != 0].sort_values(key=abs, ascending=False))
print("零:", (coef == 0).sum(), "/", len(coef))
print(coef[coef == 0].sort_values(key=abs, ascending=False))

非零: 8 / 11
rolling_mean_7    11.213595
lag_14             5.798752
lag_7              5.764174
lag_28             4.922525
is_weekend         4.711312
lag_1              0.436553
month             -0.428150
day               -0.014229
dtype: float64
零: 3 / 11
store        -0.0
item         -0.0
is_holiday   -0.0
dtype: float64


## Random Search as Ref

In [16]:
from sklearn.linear_model import ElasticNet
# ElasticNet
model4_enet = ElasticNet(
    alpha=0.026,
    l1_ratio=1,
    max_iter=20000
)

model4_enet.fit(X_train_scaled, y_train)
pred4_enet = model4_enet.predict(X_test_scaled)

print("=== Model 4b Performance: ElasticNet ===")
mae, rmse, wmape = evaluate(y_test, pred4_enet)
print(f"MAE: {mae:6.3f}   RMSE: {rmse:6.3f}   WAPE: {wmape:6.2%}")

=== Model 4b Performance: ElasticNet ===
MAE:  7.331   RMSE:  9.660   WAPE: 13.41%


In [ ]:
# Elastic Randomized Search
from sklearn.linear_model import ElasticNet
model4_enet = ElasticNet(
    alpha=0.01,
    l1_ratio=0.3,
    max_iter=20000
)

model4_enet.fit(X_train_scaled, y_train)
pred4_enet = model4_enet.predict(X_test_scaled)

print("=== Model 4b Performance: ElasticNet ===")
mae, rmse, wmape = evaluate(y_test, pred4_enet)
print(f"MAE: {mae:6.3f}   RMSE: {rmse:6.3f}   WAPE: {wmape:6.2%}")

=== Model 4b Performance: ElasticNet ===
MAE:  7.339   RMSE:  9.668   WAPE: 13.42%


## Ridge

In [17]:
from sklearn.linear_model import Ridge

# Ridge
model4_ridge = Ridge(alpha=1.0)

model4_ridge.fit(X_train_scaled, y_train)
pred4_ridge = model4_ridge.predict(X_test_scaled)

print("=== Model 4a Performance: Ridge ===")
mae, rmse, wmape = evaluate(y_test, pred4_ridge)
print(f"MAE: {mae:6.3f}   RMSE: {rmse:6.3f}   WAPE: {wmape:6.2%}")

=== Model 4a Performance: Ridge ===
MAE:  7.329   RMSE:  9.657   WAPE: 13.40%


Lasso 在交叉验证中收敛到高度稀疏解，表明模型在保留少量最相关特征的同时，仍保持了相近的预测性能。这说明短期历史销量特征是当前预测任务中的核心驱动因素，而其他背景特征的增益边际较小。

所以这里已经提升了模型的可解释性了 因为线性本来就是趋势预测（它对非线性交互的捕捉几乎是没有）我们已经可以开始解释 影响未来趋势的主要因素是什么

# Ensemble

In [23]:
# Ensemble: XGBoost + Linear reference
# 先确保前面已经训练好了：
# model_xgb, model4_ridge, model4_enet
# X_test, y_test, X_test_scaled

xgb_pred = model_xgb_default.predict(X_test)
ridge_pred = model4_ridge.predict(X_test_scaled)
enet_pred = model4_enet.predict(X_test_scaled)

ensemble_candidates = {
    "XGB": xgb_pred,
    "Ridge": ridge_pred,
    "ElasticNet": enet_pred,
    "XGB + Ridge (avg)": 0.5 * xgb_pred + 0.5 * ridge_pred,
    "XGB + ElasticNet (avg)": 0.5 * xgb_pred + 0.5 * enet_pred,
    "XGB + Ridge + ElasticNet (weighted)": 0.8 * xgb_pred + 0.1 * ridge_pred + 0.1 * enet_pred,
}

for name, pred in ensemble_candidates.items():
    mae, rmse, wmape = evaluate(y_test, pred)
    print(f"{name:28s}  MAE: {mae:6.3f}   RMSE: {rmse:6.3f}   WAPE: {wmape:6.2%}")

XGB                           MAE:  6.356   RMSE:  8.239   WAPE: 11.63%
Ridge                         MAE:  7.329   RMSE:  9.657   WAPE: 13.40%
ElasticNet                    MAE:  7.339   RMSE:  9.668   WAPE: 13.42%
XGB + Ridge (avg)             MAE:  6.593   RMSE:  8.564   WAPE: 12.06%
XGB + ElasticNet (avg)        MAE:  6.596   RMSE:  8.568   WAPE: 12.06%
XGB + Ridge + ElasticNet (weighted)  MAE:  6.375   RMSE:  8.267   WAPE: 11.66%


简单平均 ensemble 会变差，说明：线性模型的错误方向和 XGB 的错误方向并不完全一致，但它们并不能给 XGB 提供稳定增益；反而因为引入了“更弱的线性预测”，让整体误差变大；加权 ensemble（0.7 XGB + 0.15 Ridge + 0.15 ElasticNet）略好于平均 ensemble，但还是不如纯 XGB。

可是我还是觉得应该尝试集成线性啊 因为树模型比较难捕捉趋势（但显然这里它表现不错，因为lag/roll特征可能弥补了这一点）

最终路线判断：我们直接在多窗口回测的时候看看xgb和xgb+linear的效果不就好了吗 如果差不多且多次都是xgb更优的话那有点没必要了

# 业务解读

我现在的预测目标是预测每个店每种商品的sales

所以我觉得预测的精度是被预测目标限制了，这个预测目标业务直觉上来说是一个很好的目标-因为它兼容了其他你想要知道的信息，但统计学预测上来说，它不一定是就已知信息的最优选择。但如你所说，既然我们已经到了业务可接受的范围了，再做大的调整也不是很有必要，如果benchmarking没有出太大问题，我觉得这个可以就是最终方案

可以再开一个notebook换一个预测目标看一下，换一个y列的事情，问题不大